In [0]:
source_df = spark.table("bronze1.coffesales")
display(source_df)

In [0]:
%skip
if not spark.catalog.tableExists("workspace.silver1.coffesales_silver"):
    source_df.write.format("delta").option("mergeSchema", "true").saveAsTable("workspace.silver1.coffesales_silver")
else:
    source_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("workspace.silver1.coffesales_silver")

In [0]:
 %sql
select * from workspace.silver1.coffesales_silver

In [0]:
latest_silver_time = spark.sql("SELECT MAX(time) as max_time FROM workspace.silver1.coffesales_silver").collect()[0]['max_time']
latest_silver_time

In [0]:
from pyspark.sql.functions import col

bronze_new_records_df = source_df.filter(col("time") > latest_silver_time)
display(bronze_new_records_df)

In [0]:
bronze_new_records_df = bronze_new_records_df.select(
    "customer_name",
    "product",
    "quantity",
    "price",
    (col("quantity") * col("price")).alias("revenue")
)
display(bronze_new_records_df)

In [0]:
if not spark.catalog.tableExists("silver1.coffesales_silver_agg"):
    bronze_new_records_df.write.format("delta").option("mergeSchema", "true").saveAsTable("silver1.coffesales_silver_agg")
else:
    bronze_new_records_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("silver1.coffesales_silver_agg")
display(bronze_new_records_df)

In [0]:
%skip %sql
select * from workspace.silver1.coffesales_silver_agg